# How Many Dimensions Does Relevance Need? Sign-Rank and Margin Complexity

DPR proved an **upper** bound: a $d$-dimensional dual encoder realizes exactly the relevance
matrices of rank at most $d$, and Eckart–Young–Mirsky makes the truncated SVD the optimal rank-$d$
approximation to any target. But rank is the wrong complexity for a relevance **pattern**, which is
a *sign* pattern — relevant vs. not, the row-wise order, not the scores. This notebook walks the
tight question DPR deferred: how many dimensions does relevance *need*? The answer is the
**sign-rank** and its robust cousin **margin complexity**.

Every claim below is checked by an `assert` in the companion module
`embedding_dimension_lower_bounds.py`, which **owns every number** the topic page and its laboratory
display. We import it rather than reimplement anything — including DPR's finance matrix and the
in-batch loss, reused here as a byte-for-byte anchor.

In [ ]:
import pathlib, sys
# Author-time the module sits one level below notebooks/; importable by its underscored name.
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks" / "embedding-dimension-lower-bounds"))
import numpy as np
import embedding_dimension_lower_bounds as edlb

M = edlb.signed_identity(4)
print("signed identity (+1 on the diagonal, -1 off):")
print(M.astype(int))
print("rank =", int(np.linalg.matrix_rank(M)))

## Movement 1 — rank is the wrong complexity $\Rightarrow$ sign-rank

The **sign-rank** of a sign matrix $M$ is the smallest rank of a real matrix $B$ with
$\operatorname{sign}(B) = M$: the smallest embedding dimension in which the relevance *pattern* is
linearly realizable, regardless of the score magnitudes. Rank and sign-rank are genuinely different.
The signed identity has full rank $n$, yet a constructive rank-3 matrix reproduces every one of its
signs — so its sign-rank is at most $3$. The pattern needs only three dimensions; its rank says $n$.

In [ ]:
edlb.test_rank_vs_signrank_gap()
edlb.test_signrank_never_exceeds_rank()

realized, margin, B3 = edlb.realize_sign_rank(M, 3)
print("\nrank-3 realization signs match the target:",
      bool(np.all(np.sign(B3) == np.sign(M))))
print("rank of the realization:", int(np.linalg.matrix_rank(B3)))
print("sign-rank upper bound:", edlb.sign_rank_upper_bound(M), " vs  rank:", int(np.linalg.matrix_rank(M)))

## Movement 2 — a closed-form lower bound: Forster on Hadamard

Sign-rank is the right measure but intractable to compute in general. Forster's theorem gives a
*spectral* lower bound,
$$\operatorname{sign\text{-}rank}(M) \;\ge\; \frac{\sqrt{mn}}{\lVert M\rVert_2}.$$
For an $N\times N$ Hadamard matrix $H$, the rows are orthogonal, so $HH^\top = N I$ and
$\lVert H\rVert_2 = \sqrt{N}$ — the bound is exactly $\sqrt{N}$. A relevance pattern shaped like $H$
provably needs at least $\sqrt{N}$ dimensions. We confirm the spectral fact and corroborate the
bound by failing to realize $H_{16}$ in fewer than $4$ dimensions.

In [ ]:
edlb.test_forster_on_hadamard()

for k in edlb.HADAMARD_KS:
    H = edlb.hadamard_pattern(k)
    N = H.shape[0]
    print(f"N = {N:>2}:  ||H||_2 = {np.linalg.norm(H, 2):.4f} = sqrt(N) = {np.sqrt(N):.4f}; "
          f"Forster bound = {edlb.forster_lower_bound(H):.4f}")

### Eckart–Young is the wrong norm

The bridge from DPR's world: the Frobenius-optimal low-rank approximation (Eckart–Young) is *not*
the sign-optimal one. The dimension that drives the Frobenius error to zero is the full rank, but the
sign pattern is recoverable strictly below it — and at an intermediate rank the truncated SVD leaves
sign errors that a sign objective at the *same* rank fixes. Approximating in rank optimizes the wrong
quantity for a threshold pattern.

In [ ]:
edlb.test_eckart_young_is_wrong_norm()

## Movement 3 — margin complexity: the dimension for a *usable* gap

Sign-rank asks only for correct signs; entries may sit arbitrarily close to zero. **Margin
complexity** $\mathrm{mc}(M) = 1/\mathrm{margin}(M)$ asks how *robustly* the pattern separates. A
contrastively trained (soft-margin) encoder cares about the margin, not bare signs, so margin
complexity is the more faithful capacity measure. We measure the achievable normalized margin of a
correct-sign rank-$d$ realization: it is zero below the sign-rank (no correct realization exists) and
grows with $d$ above it — more dimensions buy a bigger gap.

*(We report achievable margins found by projected gradient; the exact margin complexity / $\gamma_2$
value is a semidefinite program we describe but do not solve.)*

In [ ]:
edlb.test_margin_grows_with_dimension()

print("\nmargin by embedding dimension:")
for row in edlb.margin_curve(M, edlb.MARGIN_DIMS):
    print(f"  d = {row['d']}:  realized = {row['realized']!s:>5},  margin = {row['margin']}")

## Movement 4 — the retrieval theorem and the free-embedding wall

Weller, Boratko, Naim & Lee (2025, the *LIMIT* result) make this concrete for retrieval: the minimum
embedding dimension to realize a binary qrel matrix $A$ equals the sign-rank of $2A - \mathbf{1}$ up
to $\pm 1$. The **all-pairs** qrel — one query per document pair, relevant to its two documents —
forces this to grow. Even with *perfect, freely optimized* embeddings, the largest number of
documents whose all-pairs qrel a $d$-dimensional model can realize (the *critical $n$*) grows only
polynomially in $d$. At any fixed $d$, some combinatorial relevance pattern is unrepresentable.

In [ ]:
edlb.test_free_embedding_wall()

print("\nfree-embedding critical n (largest realizable all-pairs corpus) by dimension:")
for row in edlb.critical_n_curve(edlb.CRIT_D_GRID, edlb.CRIT_N_GRID):
    print(f"  d = {row['d']}:  critical n = {row['critical_n']}")

## The reuse anchor and the finance headline flip

The in-batch InfoNCE loss is the *imported* loss, never reimplemented — on DPR's finance batch the
Gram re-derivation matches `info_nce_loss_batch` to machine precision (the same byte-for-byte anchor
DPR pins). And the headline flip: the embedding dimension that perfectly recovers single-company
retrieval on DPR's corpus **fails** combinatorial multi-company relevance over a comparable corpus,
and the combinatorial task needs strictly more dimensions.

In [ ]:
edlb.test_inbatch_loss_anchor_reused()
edlb.test_finance_headline_flip()
print()
_ = edlb.finance_demo()

## Viz constants

These are the numbers `EmbeddingDimensionLaboratory.tsx` mirrors to the decimal. The TypeScript
recomputes only closed forms (the Forster bound $\sqrt{N}$, the all-pairs query count
$\binom{n}{2}$); every measured number — the sign realization, the achievable margins, the critical-$n$
curve, the finance flip — is baked from here.

In [ ]:
edlb.viz_constants()